# KNN with DTW Baseline Model
This code implements the KNN model using DTW distance metric on our dataset. Either the full dataset or the 200 patient subset may be loaded and used for model training and evaluating. After fitting the model to our training data, we generate predictions on the validation set. Finally, we evaluate the model using various accuracy metrics.

In [23]:
pip install tslearn

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: tslearn in c:\users\lmwhe\anaconda3\envs\erdos_spring_2025\lib\site-packages (0.6.4)



In [24]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tslearn.neighbors import KNeighborsTimeSeriesClassifier
from tslearn.utils import to_time_series_dataset
from sklearn.metrics import precision_recall_curve, auc, roc_curve, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, average_precision_score, ConfusionMatrixDisplay

## Load in training, validation, and testing datasets
First we load the data into pandas dataframes. We follow this by refining the dataset to our needs, such as by excluding sparse features, normalizing, and filling in empty values.

### Full dataset

In [ ]:
path = '../../Data/Full_Dataset/'

train_df = pd.read_parquet(path+'sepsis_dataset_padded_masked_train.parquet')
val_df = pd.read_parquet(path+'sepsis_dataset_padded_masked_val.parquet')
test_df = pd.read_parquet(path+'sepsis_dataset_padded_masked_test.parquet')

Only necessary for full dataset: rename the timestep column for consistency.

In [4]:
# Rename column to ensure consistency
train_df.rename(columns={'Step': 'Hour'}, inplace=True)
val_df.rename(columns={'Step': 'Hour'}, inplace=True)
test_df.rename(columns={'Step': 'Hour'}, inplace=True)

### 200 patient subset

In [25]:
path = '../../Data/200_Pt_Subset/'

train_df = pd.read_csv(path+'subset_train.csv')
val_df = pd.read_csv(path+'subset_val.csv')
test_df = pd.read_csv(path+'subset_test.csv')

### Refine dataset
We only include relevant columns.

In [26]:
# Only include relevant features
features = ['PatientID', 'Hour', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
           #'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'PaCO2',
            'pH', 'SaO2', 'AST', 'BUN',
            #'Alkalinephos',
            'Calcium', 'Chloride', 'Creatinine', 'Glucose',
            #'Bilirubin_direct', 'Lactate',
            'Magnesium', 'Phosphate', 'Potassium',
            #'Bilirubin_total', 'TroponinI', 'PTT',
             'Hct', 'Hgb', 'WBC',
            #'Fibrinogen',
            'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
            'HospAdmTime', 'ICULOS', 'SepsisLabel']
train_df = train_df[features]
val_df = val_df[features]
test_df = test_df[features]

Define all features that we want to scale. Make a copy of the dataframe to normalize. Fit the standard scaler on training data. Obtain scaled training and valid data using transform.

In [27]:
float_feats = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
            'pH', 'SaO2', 'AST', 'BUN',
            'Calcium', 'Chloride', 'Creatinine', 'Glucose',
            'Magnesium', 'Phosphate', 'Potassium',
             'Hct', 'Hgb', 'WBC',
            'Platelets']

train_df_norm = train_df.copy()
val_df_norm = val_df.copy()
test_df_norm = test_df.copy()

# define scaler
scaler = StandardScaler()

# Fit the scaler to the data and transform it
scaler.fit(train_df_norm[float_feats])
train_df_norm[float_feats] = scaler.transform(train_df_norm[float_feats])
val_df_norm[float_feats] = scaler.transform(val_df_norm[float_feats])
test_df_norm[float_feats] = scaler.transform(test_df_norm[float_feats])

Replace NaNs with 0 and verify.

In [28]:
# Replace NaNs with 0
train_df_norm_filled = train_df_norm.fillna(0)
val_df_norm_filled = val_df_norm.fillna(0)
test_df_norm_filled = test_df_norm.fillna(0)

# check data missingness
print('train data has missing values:', train_df_norm_filled.isnull().sum().sum() != 0)
print('valid data has missing values:', val_df_norm_filled.isnull().sum().sum() != 0)
print('test data has missing values:', test_df_norm_filled.isnull().sum().sum() != 0)

train data has missing values: False
valid data has missing values: False
test data has missing values: False


## Restructure data
Define a function to get predictor/label arrays from dataframes.

In [29]:
def get_X_y(df, features_of_interest=None):
  """
  Restructure dataframe of patient time-series data into predictor and label arrays

  Parameters:
  ----------
  df : patient ICU data dataframe
  features_of_interest : features you want to include in final model; if None, includes all
                         features except PatientID, Hour, and SepsisLabel

  Returns:
  ----------
      X: np.ndarray of shape [num_patients, max_hours, num_features]
      y: np.ndarray of shape [num_patients,]
  """

  # Define which columns to use
  if features_of_interest is None:
      exclude_cols = ['PatientID', 'Hour', 'SepsisLabel']
      features_of_interest = [col for col in df.columns if col not in exclude_cols]

  # Create X array of predictors, y array of sepsis labels
  X = np.array(df[features_of_interest].values)
  y = df['SepsisLabel'].values

  return X, y

Pass the dataframes into the function and get the predictor and label arrays.

In [37]:
# Normalized, filled train/val data:
X_train, y_train = get_X_y(train_df_norm_filled)
print("X train shape:", X_train.shape)  # (num_total_timesteps, num_features)
print("y train shape:", y_train.shape)  # (num_total_timesteps,)

X_val, y_val = get_X_y(val_df_norm_filled)
print("X val shape:", X_val.shape)  # (num_total_timesteps, num_features)
print("y val shape:", y_val.shape)  # (num_total_timesteps,)

X_test, y_test = get_X_y(test_df_norm_filled)
print("X train shape:", X_test.shape)  # (num_total_timesteps, num_features)
print("y train shape:", y_test.shape)  # (num_total_timesteps,)

X train shape: (4697, 28)
y train shape: (4697,)
X val shape: (1595, 28)
y val shape: (1595,)
X train shape: (1420, 28)
y train shape: (1420,)


## KNN with DTW Model
Create a KNeighborsTimeSeriesClassifier object, and fit to the training data.

In [38]:
knn_dtw = KNeighborsTimeSeriesClassifier(n_neighbors=1, metric="dtw")
knn_dtw.fit(X_train, y_train)  # patient-level label if needed

KNeighborsTimeSeriesClassifier(n_neighbors=1)

Calculate predictions from the validation data and verify that model doesn't predict all 0's

In [ ]:
y_pred = knn_dtw.predict(X_val)
print(np.unique(y_pred))

## Model Performance
Evaluate various accuracy metrics from the model predictions. We calculate precision/recall scores, F1 score, AUROC and AUPRC, and overall accuracy. We also display the confusion matrix and plot the ROC and PR curves.

In [11]:
# Allows us to easily switch between evaluating the model on the validation or test set:

# Validation set:
X_eval = X_val 
y_true = y_val

# Test set:
# X_eval = X_test
# y_true = y_test

In [ ]:
# predictions
y_pred = knn_dtw.predict(X_eval)

# prediction probabilities
y_pred_probas = knn_dtw.predict_proba(X_eval)[:,1]

Print the accuracy metrics, including f1 score, precision, recall, AUROC, AUPRC, and overall.

In [ ]:
def get_accuracy(y_true, y_pred, y_pred_probas):
  fpr, tpr, _ = roc_curve(y_true, y_pred_probas)
  precision, recall, _ = precision_recall_curve(y_true, y_pred_probas)

  print('f1 score {0:.4f}:'.format(f1_score(y_true, y_pred)))
  print('precision {0:.4f}:'.format(precision_score(y_true, y_pred)))
  print('recall {0:.4f}:'.format(recall_score(y_true, y_pred)))
  print('AUPRC {0:.4f}:'.format(auc(recall, precision)))
  print('AUROC {0:.4f}:'.format(auc(fpr, tpr)))
  print('Overall Acc {0:.4f}:'.format(np.mean(y_pred == y_val)))

In [ ]:
get_accuracy(y_val, y_pred, y_pred_probas)

Display the confusion matrix

In [ ]:
def confusion_mat(y_true, y_pred):
  cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
  disp = ConfusionMatrixDisplay(cm, display_labels=['Non-septic', 'Septic'])
  disp.plot()

In [ ]:
confusion_mat(y_val, y_pred)

Visualisation of ROC and precision/recall curves

In [ ]:
def visualize_curves(y_true, y_pred_probas):
  # Compute metrics
  fpr, tpr, _ = roc_curve(y_val, y_pred_probas)
  roc_auc = roc_auc_score(y_val, y_pred_probas)

  precision, recall, _ = precision_recall_curve(y_val, y_pred_probas)
  pr_auc = average_precision_score(y_val, y_pred_probas)

  # Plot side-by-side
  fig, axes = plt.subplots(1, 2, figsize=(12, 5))

  # ROC curve
  axes[0].plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.3f}')
  axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
  axes[0].set_title('Receiver Operating Characteristic (ROC) Curve')
  axes[0].set_xlabel('False Positive Rate')
  axes[0].set_ylabel('True Positive Rate')
  axes[0].legend(loc='lower right')
  axes[0].grid(alpha=0.3)

  # Precision-Recall curve
  axes[1].plot(recall, precision, color='green', lw=2, label=f'AP = {pr_auc:.3f}')
  axes[1].set_title('Precision–Recall Curve')
  axes[1].set_xlabel('Recall')
  axes[1].set_ylabel('Precision')
  axes[1].legend(loc='lower left')
  axes[1].grid(alpha=0.3)

  plt.tight_layout()
  plt.show()

In [ ]:
visualize_curves(y_val, y_pred_probas)